# D2C Skincare Subscription Analytics â€” Data Simulation

## Objective

This notebook will generate a realistic synthetic dataset for users, subscriptions, orders, and marketing spend.

In [12]:
import pandas as pd
import numpy as np
from pathlib import Path

In [3]:
# Set a reproducible random seed
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [4]:
TARGET_USERS = 60_000
SIMULATION_START = "2025-01-01"
SIMULATION_END = "2027-12-31"
RAW_DATA_DIR = Path("data/raw")

# Realistic D2C skincare acquisition channels
ACQUISITION_CHANNELS = [
    "paid_search",
    "paid_social",
    "organic_search",
    "organic_social",
    "influencer",
    "referral",
    "email",
    "direct",
    "affiliate",
]

## Users Table

Generate the base users dataset with clean data (no duplicates or nulls).

In [5]:
# Generate users table
user_ids = [f"U{i:06d}" for i in range(1, TARGET_USERS + 1)]

# Generate random signup dates within simulation range
date_range = pd.date_range(start=SIMULATION_START, end=SIMULATION_END, freq="D")
signup_dates = np.random.choice(date_range, size=TARGET_USERS)

# Assign acquisition channels
acquisition_channels = np.random.choice(ACQUISITION_CHANNELS, size=TARGET_USERS)

# Assign cities (realistic D2C skincare markets)
cities = [
    "New York", "Los Angeles", "Chicago", "Houston", "Phoenix",
    "Philadelphia", "San Antonio", "San Diego", "Dallas", "San Jose",
    "Austin", "Jacksonville", "Fort Worth", "Columbus", "Charlotte",
    "San Francisco", "Indianapolis", "Seattle", "Denver", "Washington",
]
city_assignments = np.random.choice(cities, size=TARGET_USERS)

# Generate email addresses
emails = [f"user{i}@email.com" for i in range(1, TARGET_USERS + 1)]

# Create the users DataFrame
users = pd.DataFrame({
    "user_id": user_ids,
    "signup_date": signup_dates,
    "acquisition_channel": acquisition_channels,
    "city": city_assignments,
    "email": emails,
})

print("Users table generated.")

Users table generated.


In [6]:
# Validation: users table
print("=== Users Table Validation ===")
print(f"Row count: {len(users):,}")
print(f"Columns: {list(users.columns)}")
print(f"Signup date range: {users['signup_date'].min()} to {users['signup_date'].max()}")
print(f"\nAcquisition channel counts:")
print(users['acquisition_channel'].value_counts().to_string())
print(f"\nNull counts:\n{users.isnull().sum().to_string()}")
print(f"Duplicate user_ids: {users['user_id'].duplicated().sum()}")

=== Users Table Validation ===
Row count: 60,000
Columns: ['user_id', 'signup_date', 'acquisition_channel', 'city', 'email']
Signup date range: 2025-01-01 00:00:00 to 2027-12-31 00:00:00

Acquisition channel counts:
acquisition_channel
email             6762
paid_search       6758
influencer        6744
paid_social       6735
direct            6691
affiliate         6613
organic_social    6584
referral          6567
organic_search    6546

Null counts:
user_id                0
signup_date            0
acquisition_channel    0
city                   0
email                  0
Duplicate user_ids: 0


## Subscription Events

Generate realistic subscription lifecycle events: trial_start, plan_activated, paused, cancelled.

In [7]:
# Generate subscription events for each user
np.random.seed(RANDOM_SEED)

events = []

# Probabilities for subscription journey
ACTIVATION_RATE = 0.70  # 70% of trial users activate
PAUSE_RATE = 0.20       # 20% of activated users pause
CANCEL_RATE = 0.30      # 30% of activated users cancel

for idx, row in users.iterrows():
    user_id = row['user_id']
    signup_date = pd.Timestamp(row['signup_date'])
    
    # Trial starts on signup date
    trial_start = signup_date
    events.append({'user_id': user_id, 'event_date': trial_start, 'event_type': 'trial_start'})
    
    # Determine if user activates (70% chance)
    if np.random.random() < ACTIVATION_RATE:
        # Activation happens 1-14 days after trial start
        activation_delay = np.random.randint(1, 15)
        plan_activated = trial_start + pd.Timedelta(days=activation_delay)
        
        # Ensure activation is within simulation period
        if plan_activated <= pd.Timestamp(SIMULATION_END):
            events.append({'user_id': user_id, 'event_date': plan_activated, 'event_type': 'plan_activated'})
            
            # Check if user pauses (20% chance)
            if np.random.random() < PAUSE_RATE:
                # Pause happens 30-180 days after activation
                pause_delay = np.random.randint(30, 181)
                paused = plan_activated + pd.Timedelta(days=pause_delay)
                
                if paused <= pd.Timestamp(SIMULATION_END):
                    events.append({'user_id': user_id, 'event_date': paused, 'event_type': 'paused'})
            
            # Check if user cancels (30% chance)
            if np.random.random() < CANCEL_RATE:
                # Cancellation happens 14-365 days after activation
                cancel_delay = np.random.randint(14, 366)
                cancelled = plan_activated + pd.Timedelta(days=cancel_delay)
                
                if cancelled <= pd.Timestamp(SIMULATION_END):
                    events.append({'user_id': user_id, 'event_date': cancelled, 'event_type': 'cancelled'})

# Create subscription_events DataFrame
subscription_events = pd.DataFrame(events)
subscription_events['event_date'] = pd.to_datetime(subscription_events['event_date'])

print(f"Subscription events generated: {len(subscription_events):,}")

Subscription events generated: 119,325


In [8]:
# Validation: subscription events
print("=== Subscription Events Validation ===")
print(f"Total events: {len(subscription_events):,}")
print(f"Columns: {list(subscription_events.columns)}")
print(f"\nEvent type counts:")
print(subscription_events['event_type'].value_counts().to_string())

# Check for orphan user_ids
orphans = set(subscription_events['user_id']) - set(users['user_id'])
print(f"Orphan user_ids: {len(orphans)}")

# Check date range
print(f"Event date range: {subscription_events['event_date'].min()} to {subscription_events['event_date'].max()}")

# Check for null event_types
print(f"Null event_types: {subscription_events['event_type'].isnull().sum()}")

# Verify logical ordering: trial_start should come before plan_activated
trial_dates = subscription_events[subscription_events['event_type'] == 'trial_start'].set_index('user_id')['event_date']
activation_dates = subscription_events[subscription_events['event_type'] == 'plan_activated'].set_index('user_id')['event_date']
common_users = trial_dates.index.intersection(activation_dates.index)
invalid_order = (activation_dates[common_users] < trial_dates[common_users]).sum()
print(f"Invalid trial->activation order: {invalid_order}")

=== Subscription Events Validation ===
Total events: 119,325
Columns: ['user_id', 'event_date', 'event_type']

Event type counts:
event_type
trial_start       60000
plan_activated    41542
cancelled         10311
paused             7472
Orphan user_ids: 0
Event date range: 2025-01-01 00:00:00 to 2027-12-31 00:00:00
Null event_types: 0
Invalid trial->activation order: 0


## Orders

Generate realistic D2C skincare subscription orders for activated users.

In [ ]:
import pandas as pd
import numpy as np

# Generate orders for activated users
np.random.seed(RANDOM_SEED)

# Get activated users and their activation dates
activated_users = subscription_events[subscription_events['event_type'] == 'plan_activated'][['user_id', 'event_date']].copy()
activated_users.columns = ['user_id', 'activation_date']

# Plan types and their typical order values (INR)
plan_types = ['monthly', 'quarterly', 'annual']
plan_weights = [0.60, 0.25, 0.15]  # Monthly most common

# Order value ranges by plan type (INR)
value_ranges = {
    'monthly': (499, 999),
    'quarterly': (1299, 2499),
    'annual': (3999, 7999),
}

# Status distribution
statuses = ['completed', 'failed', 'refunded']
status_weights = [0.88, 0.08, 0.04]

orders = []
order_counter = 1

for _, row in activated_users.iterrows():
    user_id = row['user_id']
    activation_date = row['activation_date']
    
    # Determine if user makes repeat purchases (60% chance)
    if np.random.random() < 0.60:
        # Number of orders: 1-6
        num_orders = np.random.randint(1, 7)
    else:
        num_orders = 1
    
    for i in range(num_orders):
        # Order date: activation + 0-365 days
        order_delay = np.random.randint(0, 366)
        order_date = activation_date + pd.Timedelta(days=order_delay)
        
        # Ensure order date is within simulation period
        if order_date > pd.Timestamp(SIMULATION_END):
            continue
        
        # Select plan type
        plan_type = np.random.choice(plan_types, p=plan_weights)
        
        # Generate order value within range
        min_val, max_val = value_ranges[plan_type]
        order_value = round(np.random.uniform(min_val, max_val), 2)
        
        # Select status
        status = np.random.choice(statuses, p=status_weights)
        
        # Create order ID
        order_id = f"ORD{order_counter:07d}"
        order_counter += 1
        
        orders.append({
            'order_id': order_id,
            'user_id': user_id,
            'order_date': order_date,
            'order_value': order_value,
            'plan_type': plan_type,
            'status': status,
        })

# Create orders DataFrame
orders = pd.DataFrame(orders)
orders['order_date'] = pd.to_datetime(orders['order_date'])

print(f"Orders generated: {len(orders):,}")

Orders generated: 86,364


In [11]:
# Validation: orders
print("=== Orders Validation ===")
print(f"Row count: {len(orders):,}")
print(f"Columns: {list(orders.columns)}")
print(f"Order date range: {orders['order_date'].min()} to {orders['order_date'].max()}")
print(f"\nOrder value summary:")
print(orders['order_value'].describe().to_string())
print(f"\nOrder value <= 0 count: {(orders['order_value'] <= 0).sum()}")
print(f"\nStatus counts:")
print(orders['status'].value_counts().to_string())
print(f"\nPlan type counts:")
print(orders['plan_type'].value_counts().to_string())
print(f"\nNull counts:")
print(orders.isnull().sum().to_string())

orphans = set(orders['user_id']) - set(users['user_id'])
print(f"\nOrphan user_ids: {len(orphans)}")

=== Orders Validation ===
Row count: 86,364
Columns: ['order_id', 'user_id', 'order_date', 'order_value', 'plan_type', 'status']
Order date range: 2025-01-09 00:00:00 to 2027-12-31 00:00:00

Order value summary:
count    86364.000000
mean      1832.721508
std       1897.360777
min        499.000000
25%        707.937500
50%        915.710000
75%       2022.440000
max       7998.900000

Order value <= 0 count: 0

Status counts:
status
completed    76108
failed        6800
refunded      3456

Plan type counts:
plan_type
monthly      51936
quarterly    21273
annual       13155

Null counts:
order_id       0
user_id        0
order_date     0
order_value    0
plan_type      0
status         0

Orphan user_ids: 0


## Marketing Spend

Generate monthly marketing spend data by acquisition channel for 2025-01 through 2027-12.

In [ ]:
# Generate marketing spend data
np.random.seed(RANDOM_SEED)

# Create monthly date range for the simulation period
months = pd.date_range(start=SIMULATION_START, end=SIMULATION_END, freq='MS')

marketing_data = []

# Base spend ranges by channel (INR per month)
channel_spend_ranges = {
    'paid_search': (150000, 350000),
    'paid_social': (200000, 500000),
    'organic_search': (20000, 80000),
    'organic_social': (15000, 60000),
    'influencer': (100000, 300000),
    'referral': (30000, 100000),
    'email': (10000, 40000),
    'direct': (5000, 25000),
    'affiliate': (40000, 120000),
}

# Base new users acquired per channel per month
channel_user_ranges = {
    'paid_search': (800, 2500),
    'paid_social': (1000, 3500),
    'organic_search': (300, 1200),
    'organic_social': (200, 900),
    'influencer': (500, 2000),
    'referral': (150, 700),
    'email': (100, 500),
    'direct': (50, 300),
    'affiliate': (200, 800),
}

for month in months:
    for channel in ACQUISITION_CHANNELS:
        # Generate spend with some random variation
        min_spend, max_spend = channel_spend_ranges[channel]
        spend_inr = round(np.random.uniform(min_spend, max_spend), 2)
        
        # Generate new users acquired
        min_users, max_users = channel_user_ranges[channel]
        new_users_acquired = np.random.randint(min_users, max_users + 1)
        
        marketing_data.append({
            'month': month,
            'acquisition_channel': channel,
            'spend_inr': spend_inr,
            'new_users_acquired': new_users_acquired,
        })

# Create marketing_spend DataFrame
marketing_spend = pd.DataFrame(marketing_data)
marketing_spend['month'] = pd.to_datetime(marketing_spend['month'])

print(f"Marketing spend records generated: {len(marketing_spend):,}")

In [ ]:
# Validation: marketing spend
print("=== Marketing Spend Validation ===")
print(f"Row count: {len(marketing_spend):,}")
print(f"Columns: {list(marketing_spend.columns)}")
print(f"Month range: {marketing_spend['month'].min()} to {marketing_spend['month'].max()}")
print(f"\nNull counts:")
print(marketing_spend.isnull().sum().to_string())
print(f"\nChannel counts:")
print(marketing_spend['acquisition_channel'].value_counts().to_string())
print(f"\nSpend summary (INR):")
print(marketing_spend['spend_inr'].describe().to_string())
print(f"\nNew users summary:")
print(marketing_spend['new_users_acquired'].describe().to_string())
print(f"\nSpend < 0 count: {(marketing_spend['spend_inr'] < 0).sum()}")
print(f"New users < 0 count: {(marketing_spend['new_users_acquired'] < 0).sum()}")

## Business Signals

Add realistic, intentional differences by acquisition channel:
- funnel leak (low activation)
- worse retention (higher cancellation)
- weak LTV:CAC (low order value)

In [ ]:
# Add deterministic business signals by channelnp.random.seed(RANDOM_SEED)# Define the channels to affectFUNNEL_LEAK_CHANNEL = 'organic_social'WORSE_RETENTION_CHANNEL = 'paid_social'WEAK_LTV_CAC_CHANNEL = 'affiliate'# 1. FUNNEL LEAK: Reduce activation rate for organic_socialorganic_social_users = users[users['acquisition_channel'] == FUNNEL_LEAK_CHANNEL]['user_id'].valuesorganic_social_activations = subscription_events[    (subscription_events['user_id'].isin(organic_social_users)) &    (subscription_events['event_type'] == 'plan_activated')]activations_to_remove = organic_social_activations.sample(frac=0.60, random_state=RANDOM_SEED)subscription_events.drop(activations_to_remove.index, inplace=True)# 2. WORSE RETENTION: Increase cancellation rate for paid_socialpaid_social_users = users[users['acquisition_channel'] == WORSE_RETENTION_CHANNEL]['user_id'].valuespaid_social_activations = subscription_events[    (subscription_events['user_id'].isin(paid_social_users)) &    (subscription_events['event_type'] == 'plan_activated')]paid_social_cancelled = subscription_events[    (subscription_events['user_id'].isin(paid_social_users)) &    (subscription_events['event_type'] == 'cancelled')]['user_id'].unique()paid_social_no_cancel = paid_social_activations[~paid_social_activations['user_id'].isin(paid_social_cancelled)]extra_cancellations = paid_social_no_cancel.sample(frac=0.50, random_state=RANDOM_SEED).copy()extra_cancellations['event_type'] = 'cancelled'extra_cancellations['event_date'] = extra_cancellations['event_date'] + pd.Timedelta(days=60)subscription_events = pd.concat([subscription_events, extra_cancellations], ignore_index=True)# 3. WEAK LTV:CAC: Reduce order values for affiliate channelaffiliate_users = users[users['acquisition_channel'] == WEAK_LTV_CAC_CHANNEL]['user_id'].valuesaffiliate_mask = orders['user_id'].isin(affiliate_users)orders.loc[affiliate_mask, 'order_value'] = orders.loc[affiliate_mask, 'order_value'] * 0.45print('Business signals applied:')print(f'  Funnel leak: {FUNNEL_LEAK_CHANNEL} (removed {len(activations_to_remove)} activations)')print(f'  Worse retention: {WORSE_RETENTION_CHANNEL} (added {len(extra_cancellations)} cancellations)')print(f'  Weak LTV:CAC: {WEAK_LTV_CAC_CHANNEL} (reduced order values by 55%)')

In [ ]:
# Validation: business signalsprint('=== Business Signals Validation ===')# 1. Activation rate by channelprint('\n1. Activation Rate by Channel:')trial_counts = users.groupby('acquisition_channel')['user_id'].count()activation_by_channel = subscription_events[subscription_events['event_type'] == 'plan_activated']['user_id'].map(    users.set_index('user_id')['acquisition_channel']).value_counts()activation_rate = (activation_by_channel / trial_counts * 100).round(1)print(activation_rate.sort_values().to_string())# 2. Cancellation rate by channelprint('\n2. Cancellation Rate by Channel:')activated_by_channel = subscription_events[subscription_events['event_type'] == 'plan_activated']['user_id'].map(    users.set_index('user_id')['acquisition_channel']).value_counts()cancelled_by_channel = subscription_events[subscription_events['event_type'] == 'cancelled']['user_id'].map(    users.set_index('user_id')['acquisition_channel']).value_counts()cancellation_rate = (cancelled_by_channel / activated_by_channel * 100).round(1)print(cancellation_rate.sort_values(ascending=False).to_string())# 3. Average order value by channelprint('\n3. Average Order Value by Channel:')orders_with_channel = orders.merge(users[['user_id', 'acquisition_channel']], on='user_id')avg_order_value = orders_with_channel.groupby('acquisition_channel')['order_value'].mean().round(2)print(avg_order_value.sort_values().to_string())print('\n--- Signal Summary ---')print(f'Funnel leak: {FUNNEL_LEAK_CHANNEL}')print(f'Worse retention: {WORSE_RETENTION_CHANNEL}')print(f'Weak LTV:CAC: {WEAK_LTV_CAC_CHANNEL}')

## Messiness Injection

Inject realistic data quality issues:
- Inconsistent acquisition-channel names/casing
- ~5% null city values
- ~1% duplicate user_id rows
- Mixed event_date formats
- Some null event_type values
- Some zero/negative order_value
- Some orphan user_id values in orders
- Some missing channel-month combinations in marketing spend

In [ ]:
# Inject realistic data quality issuesnp.random.seed(RANDOM_SEED)# 1. Inconsistent acquisition-channel names/casingchannel_rename_map = {    'paid_search': 'Paid Search',    'paid_social': 'Paid_Social',}mask = users['acquisition_channel'].isin(channel_rename_map.keys())users.loc[mask, 'acquisition_channel'] = users.loc[mask, 'acquisition_channel'].map(channel_rename_map)# 2. ~5% null city valuesnull_city_idx = users.sample(frac=0.05, random_state=RANDOM_SEED).indexusers.loc[null_city_idx, 'city'] = np.nan# 3. ~1% duplicate user_id rowsdup_users = users.sample(frac=0.01, random_state=RANDOM_SEED).copy()users = pd.concat([users, dup_users], ignore_index=True)# 4. Mixed event_date formats (convert some to string with different format)date_mask = subscription_events.sample(frac=0.03, random_state=RANDOM_SEED).indexsubscription_events['event_date'] = subscription_events['event_date'].astype(object)subscription_events.loc[date_mask, 'event_date'] = subscription_events.loc[date_mask, 'event_date'].apply(lambda x: x.strftime('%d/%m/%Y') if pd.notna(x) else x)# 5. Some null event_type valuesnull_event_idx = subscription_events.sample(frac=0.02, random_state=RANDOM_SEED).indexsubscription_events.loc[null_event_idx, 'event_type'] = np.nan# 6. Some zero/negative order_valuebad_value_idx = orders.sample(frac=0.01, random_state=RANDOM_SEED).indexorders.loc[bad_value_idx[:len(bad_value_idx)//2], 'order_value'] = 0orders.loc[bad_value_idx[len(bad_value_idx)//2:], 'order_value'] = orders.loc[bad_value_idx[len(bad_value_idx)//2:], 'order_value'] * -1# 7. Some orphan user_id values in ordersorphan_orders = orders.sample(frac=0.005, random_state=RANDOM_SEED).copy()orphan_orders['user_id'] = ['ORPHAN_' + str(i).zfill(4) for i in range(len(orphan_orders))]orders = pd.concat([orders, orphan_orders], ignore_index=True)# 8. Some missing channel-month combinations in marketing spenddrop_spend = marketing_spend.sample(frac=0.02, random_state=RANDOM_SEED).indexmarketing_spend = marketing_spend.drop(drop_spend).reset_index(drop=True)print('Messiness injected:')print(f'  Channel name inconsistencies: {mask.sum()} rows')print(f'  Null city values: {len(null_city_idx)} rows')print(f'  Duplicate user rows: {len(dup_users)} rows')print(f'  Mixed date formats: {len(date_mask)} rows')print(f'  Null event_type: {len(null_event_idx)} rows')print(f'  Zero/negative order values: {len(bad_value_idx)} rows')print(f'  Orphan order user_ids: {len(orphan_orders)} rows')print(f'  Missing marketing spend rows: {len(drop_spend)} rows')

In [ ]:
# Validation: messiness injectionprint('=== Messiness Injection Validation ===')# 1. Channel name inconsistenciesprint('\n1. Acquisition Channel Names:')print(users['acquisition_channel'].value_counts().to_string())# 2. Null city valuesprint(f'\n2. Null city values: {users["city"].isnull().sum()} ({users["city"].isnull().mean()*100:.1f}%)')# 3. Duplicate user_id rowsprint(f'\n3. Duplicate user_id rows: {users["user_id"].duplicated().sum()}')# 4. Mixed date formatsprint('\n4. Event Date Format Samples:')date_types = subscription_events['event_date'].apply(type).value_counts()print(date_types.to_string())string_dates = subscription_events['event_date'].apply(lambda x: isinstance(x, str)).sum()print(f'String dates (DD/MM/YYYY): {string_dates}')print(f'Timestamp dates: {len(subscription_events) - string_dates}')print(subscription_events['event_date'].head(10).to_string())# 5. Null event_typeprint(f'\n5. Null event_type: {subscription_events["event_type"].isnull().sum()}')# 6. Zero/negative order_valueprint(f'\n6. Order value <= 0: {(orders["order_value"] <= 0).sum()}')# 7. Orphan user_ids in ordersorphan_count = orders[~orders['user_id'].isin(users['user_id'])]['user_id'].nunique()print(f'\n7. Orphan user_ids in orders: {orphan_count}')# 8. Marketing spend row countprint(f'\n8. Marketing spend rows: {len(marketing_spend):,}')print('\n--- Messiness injection complete ---')

## Final Validation and ExportValidate all tables and export to CSV files in data/raw.

In [ ]:
# Final validation and CSV exportimport osprint('=== Final Validation and CSV Export ===')# 1. Validate required columnsrequired_columns = {    'users': ['user_id', 'signup_date', 'acquisition_channel', 'city', 'email'],    'subscription_events': ['user_id', 'event_date', 'event_type'],    'orders': ['order_id', 'user_id', 'order_date', 'order_value', 'plan_type', 'status'],    'marketing_spend': ['month', 'acquisition_channel', 'spend_inr', 'new_users_acquired']}all_valid = Truefor df_name, cols in required_columns.items():    df = eval(df_name)    missing = [c for c in cols if c not in df.columns]    if missing:        print(f'FAIL: {df_name} missing columns: {missing}')        all_valid = False    else:        print(f'OK: {df_name} has all required columns')# 2. Validate users row count and duplicatesif len(users) == 60600:    print(f'OK: users has {len(users):,} rows')else:    print(f'FAIL: users has {len(users):,} rows, expected 60,600')    all_valid = Falsedup_count = users['user_id'].duplicated().sum()if dup_count == 600:    print(f'OK: users has {dup_count} duplicate user_id rows')else:    print(f'FAIL: users has {dup_count} duplicate user_id rows, expected 600')    all_valid = False# 3. Validate mixed date formats in subscription_eventsdate_types = subscription_events['event_date'].apply(type)has_timestamp = (date_types == pd.Timestamp).any() or (date_types == "<class 'pandas._libs.tslibs.timestamps.Timestamp'>").any()has_string = (date_types == str).any()if has_timestamp and has_string:    print('OK: subscription_events has both Timestamp and string event_date values')else:    print(f'FAIL: subscription_events date types: {date_types.value_counts().to_dict()}')    all_valid = False# 4. Validate messiness in ordersnon_positive = (orders['order_value'] <= 0).sum()orphan_orders = orders[~orders['user_id'].isin(users['user_id'])]['user_id'].nunique()if non_positive > 0 and orphan_orders > 0:    print(f'OK: orders has {non_positive} non-positive values and {orphan_orders} orphan user_ids')else:    print(f'FAIL: orders has {non_positive} non-positive values and {orphan_orders} orphan user_ids')    all_valid = False# 5. Validate marketing_spend row countif len(marketing_spend) == 318:    print(f'OK: marketing_spend has {len(marketing_spend)} rows')else:    print(f'FAIL: marketing_spend has {len(marketing_spend)} rows, expected 318')    all_valid = False# 6. Export to CSVos.makedirs('data/raw', exist_ok=True)users.to_csv('data/raw/users.csv', index=False)subscription_events.to_csv('data/raw/subscription_events.csv', index=False)orders.to_csv('data/raw/orders.csv', index=False)marketing_spend.to_csv('data/raw/marketing_spend.csv', index=False)print('\nCSV files exported to data/raw/')# 7. Verify output files existoutput_files = ['data/raw/users.csv', 'data/raw/subscription_events.csv', 'data/raw/orders.csv', 'data/raw/marketing_spend.csv']files_exist = all(os.path.exists(f) for f in output_files)if files_exist:    print('OK: All output files exist')else:    print('FAIL: Some output files are missing')    all_valid = False# Final resultif all_valid and files_exist:    print('\n' + '='*50)    print('LEVEL 1 COMPLETE')    print('='*50)else:    print('\nLEVEL 1 VALIDATION FAILED')